# Permutation test script

## Importing dependencies and loading data

In [ ]:
from dependencies import *

In [ ]:
df = pd.read_csv(aspect_based_sentiment_csv_filepath, encoding='utf-8-sig')

## Preparing data and parameters

In [ ]:
# defining sentiment labels
sentiment_labels   = ["Positive", "Neutral", "Negative"]

# splitting df into community groupings
legostarwars_df = df[df['subreddit'].isin(legostarwars_subs)].copy()
lego_df = df[df['subreddit'].isin(lego_subs)].copy()

communities = {
    "General LEGO subreddits":   lego_df,
    "Niche LEGO Star Wars subreddits": legostarwars_df}

In [ ]:
# setting parameters
N_PERMS      = 9999
ALPHA        = 0.05
# permutation test: statistic is difference in means (= difference in proportions)
def stat_func(x, y):
    return x.mean() - y.mean()
rng = np.random.default_rng(42)

# signifcance stars for quick visual reference
def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

def prepare_for_export(df):
    df = df.copy()
    df["sentiment"] = pd.Categorical(df["sentiment"], categories=sentiment_labels, ordered=True)
    df = df.sort_values(["aspect", "sentiment"])
    df["sentiment"] = df["sentiment"].astype(str)

    chunks = []
    for aspect, group in df.groupby("aspect", sort=False):
        chunks.append(group)
        chunks.append(pd.DataFrame([{}]))

    return pd.concat(chunks, ignore_index=True)

## Permutation tests

### Stage 1 - (dis)confirmation within communities

In [ ]:
# running stage 1 tests: permutation test for difference in proportions (p_t2 - p_t1) for each aspect × sentiment label, within each community
records = []

for community_name, community_df in communities.items():
    community_df = community_df.copy()
    community_df["aspect"] = community_df["aspect"].str.replace("_keywords", "", regex=False)
    aspects = community_df["aspect"].unique()

    for aspect in aspects:
        aspect_df = community_df[community_df["aspect"] == aspect]

        for sentiment in sentiment_labels:
            t1_labels = (aspect_df[aspect_df["time_period"].str.contains(time_periods[0])]["sentiment_label"] == sentiment).astype(int).values
            t2_labels = (aspect_df[aspect_df["time_period"].str.contains(time_periods[1])]["sentiment_label"] == sentiment).astype(int).values

            if len(t1_labels) == 0 or len(t2_labels) == 0:
                continue

            p_t1 = t1_labels.mean()
            p_t2 = t2_labels.mean()
            observed_delta = p_t2 - p_t1

            result = permutation_test(
                (t2_labels, t1_labels),
                stat_func,
                permutation_type="independent",
                n_resamples=N_PERMS,
                alternative="two-sided")

            # cohen's h for effect size
            h = proportion_effectsize(p_t2, p_t1)

            records.append({
                "community":       community_name,
                "aspect":          aspect,
                "sentiment":       sentiment,
                "transition":      f"{time_periods[0]}→{time_periods[1]}",
                "p_t1":            round(p_t1, 4),
                "p_t2":            round(p_t2, 4),
                "n_t1":            len(t1_labels),
                "n_t2":            len(t2_labels),
                "delta_p":         round(observed_delta, 4),
                "p_value":         result.pvalue,
                "cohens_h":        round(h, 4)})
stage1_results_df = pd.DataFrame(records)

# Holm correction for multiple comparisons within each community × aspect grouping (i.e. correcting across sentiment labels for each aspect within each community)
corrected_p = []
 
for (community_name, aspect), group in stage1_results_df.groupby(["community", "aspect"]):
    p_vals = group["p_value"].values
    reject, p_corrected, _, _ = multipletests(p_vals, alpha=ALPHA, method="holm")
    for i, idx in enumerate(group.index):
        corrected_p.append({
            "index":       idx,
            "p_holm":      round(p_corrected[i], 4),
            "significant": reject[i]})
 
holm_df = pd.DataFrame(corrected_p).set_index("index")
stage1_results_df = stage1_results_df.join(holm_df)

stage1_results_df["significance"] = stage1_results_df["p_holm"].apply(sig_stars)
stage1_results_df["p_value"] = stage1_results_df["p_value"].round(4)

In [ ]:
# print results
print(stage1_results_df[[
    "community", "aspect", "sentiment", "transition",
    "n_t1", "n_t2", "p_t1", "p_t2",
    "delta_p", "p_value", "p_holm", "significance", "cohens_h"]].to_string(index=False))

In [ ]:
general_df = stage1_results_df[~stage1_results_df["community"].str.contains("niche", case=False)]
niche_df   = stage1_results_df[stage1_results_df["community"].str.contains("niche", case=False)]

general_export = prepare_for_export(general_df)
niche_export   = prepare_for_export(niche_df)

general_export.to_excel("stage1_results_general.xlsx",index=False)
niche_export.to_excel("stage1_results_niche.xlsx",index=False)

### Stage 2 - difference between communities

In [ ]:
df["subreddit_grouping"] = np.where(
    df["subreddit"].isin(lego_subs),
    "General LEGO subreddits",
    np.where(
        df["subreddit"].isin(legostarwars_subs),
        "Niche LEGO Star Wars subreddits",
        None))

df = df[df["subreddit_grouping"].notna()].copy()

df["aspect"] = df["aspect"].str.replace("_keywords", "", regex=False)
aspects = df["aspect"].unique()

records = []

for aspect in aspects:
    aspect_df = df[df["aspect"] == aspect].copy()

    # binary masks for period and community grouping — computed once per aspect
    t1_mask       = aspect_df["time_period"].str.contains(time_periods[0]).values
    t2_mask       = aspect_df["time_period"].str.contains(time_periods[1]).values
    gen_mask      = aspect_df["subreddit_grouping"].values == "General LEGO subreddits"
    sw_mask       = aspect_df["subreddit_grouping"].values == "Niche LEGO Star Wars subreddits"

    for sentiment in sentiment_labels:
        # binary array: 1 if this row matches the target sentiment, 0 otherwise
        binary = (aspect_df["sentiment_label"] == sentiment).astype(int).values

        # observed proportions per community per period
        p_t1_gen = binary[gen_mask & t1_mask].mean() if (gen_mask & t1_mask).any() else 0.0
        p_t2_gen = binary[gen_mask & t2_mask].mean() if (gen_mask & t2_mask).any() else 0.0
        p_t1_sw  = binary[sw_mask  & t1_mask].mean() if (sw_mask  & t1_mask).any() else 0.0
        p_t2_sw  = binary[sw_mask  & t2_mask].mean() if (sw_mask  & t2_mask).any() else 0.0

        # observed δ_f: difference-in-differences
        # Δp for each community = sentiment proportion in t2 minus t1
        # δ_f = Δp_general - Δp_starwars
        # a positive δ_f means general LEGO shifted more than Star Wars LEGO
        delta_general  = p_t2_gen - p_t1_gen
        delta_starwars = p_t2_sw  - p_t1_sw
        observed_delta_f = delta_general - delta_starwars

        # permutation test to assess significance of observed δ_f
        null_distribution = np.empty(N_PERMS)

        grouping_labels = aspect_df["subreddit_grouping"].values

        for i in range(N_PERMS):
            shuffled_grouping = grouping_labels.copy()
            for mask in [t1_mask, t2_mask]:
                shuffled_grouping[mask] = rng.permutation(grouping_labels[mask])

            shuffled_gen_mask = shuffled_grouping == "General LEGO subreddits"
            shuffled_sw_mask  = shuffled_grouping == "Niche LEGO Star Wars subreddits"

            p_t1_gen_perm = binary[shuffled_gen_mask & t1_mask].mean() if (shuffled_gen_mask & t1_mask).any() else 0.0
            p_t2_gen_perm = binary[shuffled_gen_mask & t2_mask].mean() if (shuffled_gen_mask & t2_mask).any() else 0.0
            p_t1_sw_perm  = binary[shuffled_sw_mask  & t1_mask].mean() if (shuffled_sw_mask  & t1_mask).any() else 0.0
            p_t2_sw_perm  = binary[shuffled_sw_mask  & t2_mask].mean() if (shuffled_sw_mask  & t2_mask).any() else 0.0

            null_distribution[i] = (p_t2_gen_perm - p_t1_gen_perm) - (p_t2_sw_perm - p_t1_sw_perm)

        p_value = np.mean(np.abs(null_distribution) >= np.abs(observed_delta_f))

        records.append({
            "aspect":         aspect,
            "sentiment":      sentiment,
            "transition":     f"{time_periods[0]}→{time_periods[1]}",
            "p_t1_general":   round(p_t1_gen, 4),
            "p_t2_general":   round(p_t2_gen, 4),
            "delta_general":  round(delta_general,  4),
            "p_t1_starwars":  round(p_t1_sw,  4),
            "p_t2_starwars":  round(p_t2_sw,  4),
            "delta_starwars": round(delta_starwars, 4),
            "delta_f":        round(observed_delta_f, 4),
            "p_value":        p_value})

stage2_results_df = pd.DataFrame(records)

# Holm correction
corrected_p = []

for aspect, group in stage2_results_df.groupby("aspect"):
    p_vals = group["p_value"].values
    reject, p_corrected, _, _ = multipletests(p_vals, alpha=ALPHA, method="holm")
    for i, idx in enumerate(group.index):
        corrected_p.append({
            "index":       idx,
            "p_holm":      round(p_corrected[i], 4),
            "significant": reject[i]})

holm_df = pd.DataFrame(corrected_p).set_index("index")
stage2_results_df = stage2_results_df.join(holm_df)

stage2_results_df["significance"] = stage2_results_df["p_holm"].apply(sig_stars)
stage2_results_df["p_value"] = stage2_results_df["p_value"].round(4)

In [ ]:
# print stage 2 results
print(stage2_results_df[[
    "aspect", "sentiment", "transition",
    "delta_general", "delta_starwars", "delta_f",
    "p_value", "p_holm", "significance"]].to_string(index=False))

In [ ]:
stage2_export  = prepare_for_export(stage2_results_df)
stage2_export.to_excel("stage2_results.xlsx", index=False)